In [11]:
import shap
import spacetimeformer as stf
import sys
sys.path.append('../../bats_transformer')
from data.bats_dataset import *
from tqdm import tqdm
import numpy as np
import pandas as pd

In [12]:
ignore_cols = ["FreqLedge","AmpK@end", "Fc", "FBak15dB  ", "FBak32dB", "EndF", "FBak20dB", "LowFreq", "Bndw20dB", 
               "CallsPerSec", "EndSlope", "SteepestSlope", "StartSlope", "Bndw15dB", "HiFtoUpprKnSlp", "HiFtoKnSlope", 
               "DominantSlope", "Bndw5dB", "PreFc500", "PreFc1000", "PreFc3000", "KneeToFcSlope", "TotalSlope", 
               "PreFc250", "CallDuration", "CummNmlzdSlp", "DurOf32dB", "SlopeAtFc", "LdgToFcSlp", "DurOf20dB", "DurOf15dB", 
               "TimeFromMaxToFc", "KnToFcDur", "HiFtoFcExpAmp", "AmpKurtosis", "LowestSlope", "KnToFcDmp", "HiFtoKnExpAmp", 
               "DurOf5dB", "KnToFcExpAmp", "RelPwr3rdTo1st", "LnExpB_StartAmp", "Filter", "HiFtoKnDmp", "LnExpB_EndAmp", 
               "HiFtoFcDmp", "AmpSkew", "LedgeDuration", "KneeToFcResidue", "PreFc3000Residue", "AmpGausR2", "PreFc1000Residue", 
               "Amp1stMean", "LdgToFcExp", "FcMinusEndF", "Amp4thMean", "HiFtoUpprKnExp", "HiFtoKnExp", "KnToFcExp", "UpprKnToKnExp", 
               "Kn-FcCurviness", "Amp2ndMean", "Quality", "HiFtoFcExp", "LnExpA_EndAmp", "RelPwr2ndTo1st", "LnExpA_StartAmp", 
               "HiFminusStartF", "Amp3rdMean", "PreFc500Residue", "Kn-FcCurvinessTrndSlp", "PreFc250Residue", "AmpVariance", "AmpMoment", 
               "meanKn-FcCurviness", "MinAccpQuality", "AmpEndLn60ExpC", "AmpStartLn60ExpC", "Preemphasis", "MaxSegLnght" ,"Max#CallsConsidered" ]
ignore_cols += ["Filename", "NextDirUp", 'Path', 'Version', 'Filter', 'Preemphasis', 'MaxSegLnght', "ParentDir", "file_id", "chirp_idx", "split"]

In [13]:
data_module = stf.data.DataModule(
    datasetCls = BatsCSVDataset,
    dataset_kwargs = {
        "root_path": "../../bats_transformer/data/july_daytime_chunked_quantile/splits",
        "prefix": "split",
        "ignore_cols": ignore_cols,
        "time_col_name": "TimeIndex",
        "val_split": 0.05,
        "test_split": 0.05,
        "context_points": None,
        "target_points": 1,
        "random_seed": 31
    },
    batch_size=64,
    workers=4,
)

In [14]:
train_data = data_module.train_dataloader()
val_data = data_module.val_dataloader()
test_data = data_module.test_dataloader()

Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations


In [24]:
truths = []
preds = []
errors = []

for batch in tqdm(test_data):
    x_t, x_c, y_t, y_c = batch
    mask = x_t > 0
    lengths = mask.sum(dim=1)
    feature_sums = x_c.sum(dim=1)
    for i, row in enumerate(feature_sums):
        pred = x_c[i, -1, :]
        truths.append(y_c[i].numpy()[0])
        preds.append(pred.numpy())
        errors.append((y_c[i] - pred).numpy()[0])

truths, preds, errors = np.array(truths), np.array(preds), np.array(errors)

100%|██████████| 22/22 [00:05<00:00,  4.05it/s]


In [ ]:
target_columns = train_data.dataset.target_cols
pd.DataFrame(preds)
# target_columns

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,0.839121,-0.645631,0.260759,0.533871,0.006765,0.085789,-1.302875,2.513015,0.271330,0.781580,...,-0.533953,-0.732192,-1.254076,1.015744,-1.549900,-0.433581,0.083062,-0.284112,0.159982,0.512483
1,0.758004,0.465405,1.507399,1.206667,1.445984,-0.674706,1.761428,-1.259755,1.550054,1.136886,...,-0.084978,0.347918,0.440610,-0.888543,0.227728,1.335837,1.531678,0.977207,1.065919,-1.005047
2,0.607463,1.044409,1.854786,1.990070,1.088106,-0.331979,1.672414,-0.884548,1.899486,1.024058,...,0.413532,-0.325384,-0.494522,0.741277,0.444240,1.197589,1.105417,2.590991,2.347429,-0.835938
3,0.293986,0.661162,1.906924,2.017829,0.740709,-0.020545,1.552670,-0.710632,1.961262,0.875705,...,1.386384,-0.759530,-0.012026,0.331293,-0.407695,1.463375,1.567807,0.950771,0.947218,-0.655856
4,0.123259,-0.090452,1.903590,2.002110,0.837079,-0.219158,1.672216,-0.901279,1.957133,1.023805,...,0.253661,-0.888444,-0.239253,0.561857,-0.024111,1.425481,1.449124,0.819541,0.747003,-0.859159
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1377,0.969427,1.212751,-0.665146,-0.588628,-0.043029,-0.344211,-0.241242,0.188005,-0.639753,-0.874316,...,0.807859,1.513985,2.111374,1.635191,0.864304,0.619084,0.461818,0.835518,0.648480,0.339913
1378,0.608973,-0.684763,0.231114,-0.381582,0.436725,-0.162884,0.787606,-1.675788,0.237989,0.057177,...,-1.224635,0.018295,0.610624,0.189168,0.849243,0.354388,-1.035347,-0.044595,-1.031763,-1.720743
1379,0.388430,0.864365,1.158272,0.826709,0.614529,0.149449,0.216940,0.688360,1.200187,0.802330,...,0.873541,-0.035907,-0.136750,0.250450,-0.037054,0.725857,0.856949,0.547525,0.531275,-0.678142
1380,0.121995,1.044409,1.292374,0.730989,0.429126,2.150108,0.372726,1.107677,1.319502,1.758515,...,0.414780,0.109993,-0.540859,-0.233849,0.072738,0.242215,0.578859,2.544454,2.524433,2.350017


In [27]:
mae = np.abs(errors).mean(axis=0)
mse = (errors * errors).mean(axis=0)

In [28]:
mse_df = pd.DataFrame(np.array([target_columns, mse]).T)
mse_df = mse_df.set_index(0)
# mse_df[1] = mse_df[1].round(6)
pd.Series(mse, index=target_columns)

TimeInFile          0.191320
PrecedingIntrvl     1.090260
HiFreq              1.103198
Bndwdth             1.421280
FreqMaxPwr          0.773459
PrcntMaxAmpDur      1.656519
FreqKnee            1.139879
PrcntKneeDur        1.427251
StartF              1.134904
UpprKnFreq          1.254696
HiFtoUpprKnAmp      1.061287
HiFtoKnAmp          1.483262
HiFtoFcAmp          1.103981
UpprKnToKnAmp      14.987539
KnToFcAmp           1.295417
LdgToFcAmp          1.550591
FreqCtr             0.850822
FFwd32dB            0.986857
FFwd20dB            0.999570
FFwd15dB            1.041959
FBak5dB             0.717396
FFwd5dB             0.886651
Bndw32dB            0.922224
Amp1stQrtl          1.337686
Amp2ndQrtl          1.238118
Amp3rdQrtl          1.247284
Amp4thQrtl          1.557255
1st10kHzSlp         1.267033
1st5to15kHzSlp      1.890953
1st10kHzExp         1.133779
1st5to15kHzExp      1.791266
AmpK@start          1.627225
dtype: float32

In [37]:
average_loss_per_row = mse_df.mean(axis=1)
average_loss_per_row_no_outlier = mse_df.drop("UpprKnToKnAmp", axis=0).mean(axis=1)
print(average_loss_per_row.mean(), average_loss_per_row_no_outlier.mean())

1.6303412965624997 1.1994639512903225
